In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

matrix = pd.read_pickle("cache/matrix.pkl")
cluster_features = pd.read_pickle('cache/cluster_features.pkl')
matrix = pd.merge(matrix, cluster_features, on=['item_id'], how='left')
matrix['cluster'] = matrix['cluster'].fillna(-1).astype(np.int8)

test_clean = pd.read_pickle("cache/test_clean.pkl")
print("Loaded matrix:", matrix.shape, "test_clean:", test_clean.shape)

X_train = matrix[matrix.date_block_num < 33].drop(columns=["item_cnt_month"])   
Y_train = matrix[matrix.date_block_num < 33]["item_cnt_month"]                  
X_valid = matrix[matrix.date_block_num == 33].drop(columns=["item_cnt_month"])
Y_valid = matrix[matrix.date_block_num == 33]["item_cnt_month"]
X_test  = matrix[matrix.date_block_num == 34].drop(columns=["item_cnt_month"])

def categorize_sales(x):
    if x ==0:
        return 'no_sales'
    elif x <= 5:
        return 'low'
    elif x <= 15:
        return 'medium'
    else:
        return 'high'

Y_train_cls = Y_train.apply(categorize_sales)
Y_valid_cls = Y_valid.apply(categorize_sales)

le = LabelEncoder()
Y_train_enc = le.fit_transform(Y_train_cls)
Y_valid_enc = le.transform(Y_valid_cls)

print("train class ratio:\n", Y_train_cls.value_counts(normalize=True))
print("valid class ratio:\n", Y_valid_cls.value_counts(normalize=True))

clf = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.03,
    min_child_weight=300,
    colsample_bytree=0.8,
    subsample=0.8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

clf.fit(
    X_train,
    Y_train_enc,
    eval_set=[(X_valid, Y_valid_enc)],
    verbose=True,
)

Y_pred_enc = clf.predict(X_valid)
Y_pred = le.inverse_transform(Y_pred_enc)

print(classification_report(Y_valid_cls, Y_pred))
print(confusion_matrix(Y_valid_cls, Y_pred))

test_pred_enc = clf.predict(X_test)
test_pred_cls = le.inverse_transform(test_pred_enc)

# 予測値と (shop_id, item_id) をセットにした DF を作る
pred34 = (
    matrix[matrix['date_block_num'] == 34][['shop_id', 'item_id']]
    .copy()
    .assign(sales_class=test_pred_cls)
)

# test_clean を使って安全にマージ
submission_cls = (
    test_clean
    .merge(pred34, on=['shop_id', 'item_id'], how='left')
    .set_index('ID')[['sales_class']]
)

# 欠損チェック（これで0になるはず）
missing = submission_cls['sales_class'].isna().sum()
print("NaNs in submission:", missing)

if missing > 0:
    print(f"⚠️  Warning: {missing} NaNs found in submission!")
    majority_class = Y_train_cls.mode().iloc[0]
    submission_cls['sales_class'] = submission_cls['sales_class'].fillna(majority_class)
    print(f"✅ Filled {missing} NaNs with majority class: {majority_class}")
    
    # デバッグモードの場合はassertでも停止可能
    # assert False, f"予測に{missing}個の欠損があります。データを確認してください。"

# 最終チェック: 修正後も欠損が残っていたら確実にエラー
assert submission_cls['sales_class'].isna().sum() == 0, "修正後も欠損が残っています。"

submission_cls.to_csv("submission_classification_improved.csv")
print("Saved: submission_classification_improved.csv")
submission_cls.head()



#クラスタリングを利用して精度の改善を試みましたが、向上していない可能性が高いです。

Loaded matrix: (11128004, 14) test_clean: (214200, 3)
train class ratio:
 item_cnt_month
no_sales    0.852547
low         0.138913
medium      0.006866
high        0.001674
Name: proportion, dtype: float64
valid class ratio:
 item_cnt_month
no_sales    0.867864
low         0.125409
medium      0.005223
high        0.001503
Name: proportion, dtype: float64
[0]	validation_0-mlogloss:1.34229
[1]	validation_0-mlogloss:1.30054
[2]	validation_0-mlogloss:1.26178
[3]	validation_0-mlogloss:1.22475
[4]	validation_0-mlogloss:1.19012
[5]	validation_0-mlogloss:1.15690
[6]	validation_0-mlogloss:1.12540
[7]	validation_0-mlogloss:1.09593
[8]	validation_0-mlogloss:1.06740
[9]	validation_0-mlogloss:1.04040
[10]	validation_0-mlogloss:1.01507
[11]	validation_0-mlogloss:0.99069
[12]	validation_0-mlogloss:0.96747
[13]	validation_0-mlogloss:0.94491
[14]	validation_0-mlogloss:0.92327
[15]	validation_0-mlogloss:0.90251
[16]	validation_0-mlogloss:0.88271
[17]	validation_0-mlogloss:0.86378
[18]	validation_0-mlog

,sales_class
ID,
0,no_sales
1,no_sales
2,low
3,no_sales
4,no_sales
